Đây là script chạy huấn luyện mô hình đề xuất gồm các bước:
- Clone repository từ GitHub
- Load dữ liệu đề bài của cuộc thi
- Huấn luyện mô hình
- Lưu kết quả huấn luyện và file submission

**Lưu ý**:
- Script này **không thể** chạy local do ngốn bộ nhớ. Script **bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Notebook này là notebook chạy stage 2 trong quá trình huấn luyện (Huấn luyện mô hình machine learning từ embedding của mô hình deep learning), notebook chạy stage 1 có thể xem tại [đây](./dataflow2026_hd4k_run_stage_1.ipynb).
- Notebook này cho phép chạy 3 mô hình mà nhóm đề xuất: Chrono-C, Chrono-G và Chrono-R. Mỗi mô hình sẽ kết hợp với 2 mô hình cây nữa là LightGBM và XGBoost, tổng cộng sẽ có 6 mô hình được huấn luyện.
- Thời gian chạy của cả 6 mô hình đều tối đa 2 tiếng, chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle.
- Cell code 10 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)
    - Chọn loại mô hình muốn chạy

In [ ]:
!git clone https://github.com/CryAndRRich/chronolens.git

In [ ]:
%cd /kaggle/working/chronolens
CHRONO_PATH = "/kaggle/working/chronolens"

In [ ]:
!pip install -r requirements.txt

In [4]:
import sys

if CHRONO_PATH not in sys.path:
    sys.path.append(CHRONO_PATH)

In [5]:
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [6]:
from config import *
from preprocess import *
from model import *
from utils import *

In [7]:
from model.train import train_tree_model

In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [9]:
INPUT_ROOT = "YOUR_INPUT_ROOT_PATH"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"

EMBEDDING_NAME = "chrono_g"
MODEL_NAME = "lgbm"
CHECKPOINT_FILE = f"{INPUT_ROOT}/weights/best_model_{EMBEDDING_NAME}.pth"
ATTRIBUTE_LIST = [1, 2, 3, 4, 5, 6]
TARGET_INDICES = [1, 2, 3, 5, 6, 7]
N_TRIALS = 1

In [10]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [ ]:
model_kwargs = update_model_kwargs(
    data_manager=data_manager,
    model_kwargs=CONFIG_MODEL.MODEL_KWARGS[EMBEDDING_NAME]
)
model = get_model(
    name=EMBEDDING_NAME,
    **model_kwargs  
).to(DEVICE)

model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

In [12]:
x_train, y_train, x_val, y_val, x_test = data_manager.get_embedding(model, DEVICE)
y_true = y_val[:,[1,2,3,5,6,7]]

In [ ]:
val_pred = np.zeros((len(x_val), 6), dtype=np.float32)
test_pred = np.zeros((len(x_test), 6), dtype=np.float32)

x_full = np.concatenate([x_train, x_val], axis=0)

dval = xgb.DMatrix(x_val, label=y_val) if MODEL_NAME == "xgb" else None
dtest = xgb.DMatrix(x_test) if MODEL_NAME == "xgb" else None

for i, target_idx in enumerate(TARGET_INDICES):
    attr_name = ATTRIBUTE_LIST[i]
    
    y_train_target = y_train[:, target_idx]
    y_val_target = y_val[:, target_idx]

    y_full_target = np.concatenate([y_train_target, y_val_target], axis=0)
    
    search_model, final_model = train_tree_model(
        model_name=MODEL_NAME,
        x_train=x_train, 
        y_train=y_train_target, 
        x_val=x_val,
        y_val=y_val_target, 
        x_full=x_full,
        y_full=y_full_target,
        attr_name=attr_name,
        n_trials=N_TRIALS,
        random_seed=RANDOM_SEED
    )

    if dval and dtest:
        val_pred[:, i] = search_model.predict(dval, iteration_range=(0, search_model.best_iteration + 1))
        test_pred[:, i] = final_model.predict(dtest, iteration_range=(0, final_model.best_iteration + 1))
    else:
        val_pred[:, i] = search_model.predict(x_val, num_iteration=search_model.best_iteration)
        test_pred[:, i] = final_model.predict(x_test, num_iteration=final_model.best_iteration)

In [ ]:
get_stats(y_true, val_pred)

In [ ]:
test_pred = torch.tensor(test_pred, device=DEVICE, dtype=torch.float32)
test_pred = post_process_predictions(test_pred).cpu().numpy()

submission_df = pd.DataFrame({"id": data_manager.x_test["id"]})
for idx, attr in enumerate(ATTRIBUTE_LIST):
    submission_df[f"attr_{attr}"] = test_pred[:, idx].astype(np.uint16)

submission_df.to_csv(SUBMISSION_FILE, index=False)
print(submission_df)
print(submission_df.dtypes)